In [1]:
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Cipher import AES, PKCS1_OAEP

import socket
import ssl
import time

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 40005

data = "Menda bo novo glavno mesto Novo Novo mesto.".encode("utf-8")

if __name__ == "__main__":
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client.connect((SERVER_HOST, SERVER_PORT))

    recipient_key = RSA.import_key(open("receiver.pem").read())
    session_key = get_random_bytes(16)
    
    cipher_rsa = PKCS1_OAEP.new(recipient_key)
    enc_session_key = cipher_rsa.encrypt(session_key)
    
    client.sendall(enc_session_key)
    
    while True:
        data = input('Kaj pošljemo: ')
        data = data.encode('utf-8')
        
        cipher_aes = AES.new(session_key, AES.MODE_EAX)
        ciphertext, tag = cipher_aes.encrypt_and_digest(data)
        
        client.sendall(cipher_aes.nonce)
        client.sendall(tag)
        client.sendall(ciphertext)
        print('Sent!')

Kaj pošljemo: halo
Sent!


KeyboardInterrupt: Interrupted by user